# Modelado con K-Fold Cross Validation y Ajuste de Hiperparámetros

Este notebook implementa las mejores prácticas para la evaluación de modelos de Machine Learning utilizando **K-Fold Cross Validation** (CV). El objetivo es encontrar la configuración óptima para cada modelo y obtener una estimación robusta de su desempeño antes de probar con los datos finales de prueba.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [ ]:
# Cargar los datos procesados con PCA
X_train = np.load('X_train_pca.npy')
X_test = np.load('X_test_pca.npy')
y_train = np.load('y_train.npy')
y_test = np.load('y_test.npy')

print("Datos cargados exitosamente.")
print(f"Entrenamiento: {X_train.shape}")
print(f"Prueba (Hold-out): {X_test.shape}")

---
## 1. Configuración de la Estrategia de Validación

Definimos el objeto `StratifiedKFold` en un bloque separado. Esto asegura que todos los modelos que evaluemos a continuación sean probados sobre las **mismas particiones de datos**, permitiendo una comparación científica y justa.

In [ ]:

# Utilizamos Stratified para mantener el balance de clases (65% Benigno / 35% Maligno) en cada fold.
cv_strategy = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

print(f"Estrategia configurada: {cv_strategy.n_splits} Folds Estratificados.")

---
## 2. Ajuste de Hiperparámetros (Grid Search)

Utilizaremos `GridSearchCV` para encontrar los mejores parámetros. Optimizaremos para la métrica **F1-Score** (o Recall) para asegurar que el modelo sea sensible a la detección de tumores malignos.

In [ ]:
print("Sintonizando SVM...")
param_grid_svm = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.01, 0.1, 1],
    'kernel': ['rbf', 'linear']
}

grid_svm = GridSearchCV(SVC(probability=True, class_weight='balanced'), 
                        param_grid_svm, 
                        cv=cv_strategy, 
                        scoring='f1', 
                        verbose=1)
grid_svm.fit(X_train, y_train)

print(f"Mejores parámetros SVM: {grid_svm.best_params_}")
print(f"Mejor F1-Score (CV): {grid_svm.best_score_:.4f}")

In [ ]:
print("\nSintonizando Regresión Logística...")
param_grid_lr = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l2'],
    'solver': ['lbfgs', 'liblinear']
}

grid_lr = GridSearchCV(LogisticRegression(max_iter=1000, class_weight='balanced'), 
                       param_grid_lr, 
                       cv=cv_strategy, 
                       scoring='f1', 
                       verbose=1)
grid_lr.fit(X_train, y_train)

print(f"Mejores parámetros LR: {grid_lr.best_params_}")
print(f"Mejor F1-Score (CV): {grid_lr.best_score_:.4f}")

In [ ]:
print("\nSintonizando Random Forest...")
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5]
}

grid_rf = GridSearchCV(RandomForestClassifier(class_weight='balanced', random_state=42), 
                       param_grid_rf, 
                       cv=cv_strategy, 
                       scoring='f1', 
                       verbose=1)
grid_rf.fit(X_train, y_train)

print(f"Mejores parámetros RF: {grid_rf.best_params_}")
print(f"Mejor F1-Score (CV): {grid_rf.best_score_:.4f}")

---
## 3. Evaluación Final en el Conjunto de Prueba (Unseen Data)

Seleccionamos el mejor estimador basado en los resultados de Cross-Validation y lo validamos con el conjunto `X_test` que no fue utilizado en ninguna parte del proceso de sintonización.

In [ ]:
best_model = grid_svm.best_estimator_

y_pred = best_model.predict(X_test)
print("Reporte de Clasificación Final (Sobre Test Set)")
print("="*45)
print(classification_report(y_test, y_pred, target_names=['Benigno', 'Maligno']))

# Matriz de Confusión
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens')
plt.title('Matriz de Confusión Final')
plt.xlabel('Predicho')
plt.ylabel('Real')
plt.show()